# Flow Matching: Unconstrained Optimization

Find stationary points (minima, maxima) by conditioning on $f'(x) = 0$.

**Authors:** Victor Alves and John R. Kitchin

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU for JAX (must be set before importing JAX)
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    cluster_stats,
    ConditionalFlowMatching
)

# Force CPU for PyTorch
device = torch.device('cpu')
print(f"Using device: {device}")

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Problem Setup

For $f(x) = x^4 - 4x^2 + x + 3$, find stationary points where $f'(x) = 0$.

The derivative is $f'(x) = 4x^3 - 8x + 1$.

The approach:
1. Generate training data: pairs of $(x, f'(x))$
2. Train flow matching to generate $x$ conditioned on $f'(x)$
3. Sample by conditioning on $f'(x) = 0$

In [ ]:
# Generate data with derivatives - more points and oversample near stationary points
x_base = np.linspace(-2.25, 2, 1000)
y_base = x_base**4 - 4*x_base**2 + x_base + 3
y_prime_base = 4*x_base**3 - 8*x_base + 1

# True stationary points
stationary = np.roots([4, 0, -8, 1])
print(f"Stationary points: {stationary}")

# Oversample near stationary points (where y' is close to 0)
x_near_stat = []
for s in stationary:
    x_near_stat.append(np.linspace(s - 0.3, s + 0.3, 200))
x_near_stat = np.concatenate(x_near_stat)
y_prime_near_stat = 4*x_near_stat**3 - 8*x_near_stat + 1

# Combine base and oversampled data
x = np.concatenate([x_base, x_near_stat])
y_prime = np.concatenate([y_prime_base, y_prime_near_stat])

print(f"Total data points: {len(x)}")
print(f"Data points near stationary points: {len(x_near_stat)}")

In [ ]:
# Train flow matching: generate x conditioned on y'
# Use larger model and more epochs
x_data = x.reshape(-1, 1)
c_data = y_prime.reshape(-1, 1)

fm_opt = ConditionalFlowMatching(x_dim=1, c_dim=1, hidden_dim=128, n_layers=4, sigma_min=0.001)
losses = fm_opt.fit(x_data, c_data, epochs=1000, batch_size=128)

In [ ]:
# Sample stationary points by conditioning on y'=0
# Use more integration steps
samples = fm_opt.sample(c_values=[[0.0]], n_samples=1000, n_steps=100)

plt.figure(figsize=(8, 4))
plt.hist(samples[:, 0], bins=50, density=True, alpha=0.7)
for s in stationary:
    plt.axvline(s, color='r', ls='--', lw=2)
plt.xlabel('x')
plt.ylabel('Density')
plt.title("Stationary points (conditioned on $f'(x)=0$)")
plt.grid(True, alpha=0.3)
plt.show()

print("Cluster statistics:")
cluster_stats(samples, eps=0.3)

## Distinguishing Minima from Maxima

We can use the sign of the second derivative to distinguish:
- $f''(x) > 0$: minimum
- $f''(x) < 0$: maximum

In [ ]:
# Distinguish minima from maxima using second derivative sign
y_double_prime_sign = np.where((12 * x**2 - 8) > 0, 1.0, -1.0)

# Condition on both y' and sign(y'')
c_data = np.column_stack([y_prime, y_double_prime_sign])

fm_opt2 = ConditionalFlowMatching(x_dim=1, c_dim=2, hidden_dim=128, n_layers=4, sigma_min=0.001)
losses = fm_opt2.fit(x_data, c_data, epochs=1000, batch_size=128)

In [ ]:
# Find maximum: y'=0, y''<0 (sign=-1)
samples_max = fm_opt2.sample(c_values=[[0.0, -1.0]], n_samples=500, n_steps=100)

print("Maximum (y'=0, y''<0):")
cluster_stats(samples_max, eps=0.3)

# Find minima: y'=0, y''>0 (sign=+1)
samples_min = fm_opt2.sample(c_values=[[0.0, 1.0]], n_samples=500, n_steps=100)

print("\nMinima (y'=0, y''>0):")
cluster_stats(samples_min, eps=0.3)

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(samples_max[:, 0], bins=30, alpha=0.7)
axes[0].axvline(0.126, color='r', ls='--', lw=2, label='True max')
axes[0].set_xlabel('x')
axes[0].set_title('Maximum location')
axes[0].legend()

axes[1].hist(samples_min[:, 0], bins=30, alpha=0.7)
axes[1].axvline(-1.473, color='r', ls='--', lw=2, label='True min')
axes[1].axvline(1.347, color='r', ls='--', lw=2)
axes[1].set_xlabel('x')
axes[1].set_title('Minima locations')
axes[1].legend()

plt.tight_layout()
plt.show()